# 10 · SVM visualisation

This notebook is the **educational part** of lab 10. It follows the lecture on support vector machines: the maximum margin,
the soft margin and parameter C, the hinge loss, kernels and parameter γ, scaling, regression with the ε-tube and the one-class SVM.
Every method runs on small synthetic data and has a plot with **controls** (sliders, menus, check boxes). You do not need to read or change any code.

**How to work with it:** run all cells (*Run → Run All Cells*). Under every part, controls appear above its plot; each change redraws
the plot. The code cells are collapsed; click the bar to the left of a cell to expand one if you are curious.
Each part ends with a **✏️ Task**; your answers will help you in the practical part (`svm-practice-en.ipynb`).

If no controls appear, the package `ipywidgets` is missing (see the README).

In [ ]:
import numpy as np                          # arrays and maths
import pandas as pd                         # result tables
import matplotlib.pyplot as plt             # plots
import ipywidgets as widgets                # sliders, menus and check boxes
from IPython.display import display         # tables
from sklearn.svm import SVC, SVR, OneClassSVM          # SVM for classification, regression and one class
from sklearn.model_selection import cross_val_score    # cross-validation
from sklearn.preprocessing import StandardScaler       # standardisation
from sklearn.pipeline import make_pipeline             # scaling followed by a model
from sklearn.datasets import make_moons, make_circles  # small synthetic data sets

BLUE, ORANGE, TEAL, GREY = "#0065BD", "#C2410C", "#00897B", "#8C96A0"  # colours as in the lecture
PALE_B, PALE_O = "#DCE9F6", "#F8E3D9"       # pale class regions
plt.rcParams.update({"figure.figsize": (11, 4.2), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})
STYLE = {"description_width": "190px"}      # room for the control labels
WIDE = widgets.Layout(width="95%", max_width="560px")


def fslider(label, value, lo, hi, step, fmt=".2f"):
    return widgets.FloatSlider(value=value, min=lo, max=hi, step=step, description=label, readout_format=fmt,
                               continuous_update=False, style=STYLE, layout=WIDE)


def logslider(label, value, lo, hi):
    # slider on a logarithmic scale: lo and hi are powers of ten
    return widgets.FloatLogSlider(value=value, base=10, min=lo, max=hi, step=0.25, description=label,
                                  readout_format=".3g", continuous_update=False, style=STYLE, layout=WIDE)


def menu(label, options, value):
    return widgets.Dropdown(options=options, value=value, description=label, style=STYLE, layout=WIDE)


def check(label, value=False):
    return widgets.Checkbox(value=value, description=label, style=STYLE, layout=WIDE)


def controls(draw, **inputs):
    # shows the controls and redraws the plot after every change
    widgets.interact(draw, **inputs)


def show_classes(ax, X, y, s=40, alpha=1.0):
    # class +1 blue circles, class −1 orange triangles, as in the lecture
    p = y == 1
    ax.scatter(X[p, 0], X[p, 1], s=s, color=BLUE, edgecolor="white", lw=0.6, alpha=alpha, zorder=3, label="class +1")
    ax.scatter(X[~p, 0], X[~p, 1], s=s * 1.1, marker="^", color=ORANGE, edgecolor="white", lw=0.6, alpha=alpha, zorder=3, label="class −1")


def ring(ax, P, color="black", s=200, label=None):
    ax.scatter(P[:, 0], P[:, 1], s=s, facecolors="none", edgecolors=color, lw=1.8, zorder=4, label=label)


def draw_regions(ax, model, xlim, ylim, margins=True, n=250):
    # class regions, boundary (score 0) and margin edges (score ±1)
    xx, yy = np.meshgrid(np.linspace(*xlim, n), np.linspace(*ylim, n))
    zz = model.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, zz, levels=[-1e9, 0, 1e9], colors=[PALE_O, PALE_B], alpha=0.8)
    ax.contour(xx, yy, zz, levels=[0], colors="black", linewidths=2.2)
    if margins:
        ax.contour(xx, yy, zz, levels=[-1, 1], colors="black", linewidths=1.2, linestyles="--")


def blobs(seed, n, spread, c=1.3):
    g = np.random.default_rng(seed)
    X = np.r_[g.normal(c, spread, (n, 2)), g.normal(-c, spread, (n, 2))]
    return X, np.r_[np.ones(n), -np.ones(n)].astype(int)


print("Ready. Run the following cells; controls will appear under each part.")

## 1. Perceptron versus maximum margin

**Goal of the method:** among all lines without a training error choose the one farthest from the nearest points. The distance of
a point from the line w·x + b = 0 is d = y·(w·x + b)/‖w‖; the **margin** is the smallest d. The points that define it are the **support vectors**.

The data are the four points from lectures 09 and 10. The perceptron stops at the first error-free line, which depends on the order of the samples.
SVM gives w = (0.8, 0.4), b = −1 and the margin width 2/‖w‖ = √5 ≈ 2.24 regardless of the order.

**Controls:** the order of samples for the perceptron; the check box adds a fifth point of class +1 whose position you move with the sliders.

In [ ]:
P1X = np.array([[3, 2], [0, -2], [1, 3], [-1, 2]], dtype=float)  # the four points from the lecture
P1Y = np.array([1, -1, 1, -1])


def perceptron(X, y, order, epochs=50):
    w, b = np.zeros(2), 0.0
    for _ in range(epochs):
        errors = 0
        for i in order:
            if y[i] * (w @ X[i] + b) <= 0:          # error: move the line towards the sample
                w, b, errors = w + y[i] * X[i], b + y[i], errors + 1
        if errors == 0:
            break
    return w, b


def line(ax, w, b, xlim, **kw):
    xs = np.array(xlim, dtype=float)
    if abs(w[1]) > 1e-12:
        ax.plot(xs, -(w[0] * xs + b) / w[1], **kw)
    else:
        ax.axvline(-b / w[0], **kw)


def section1(order, add_point, px, py):
    X, y = P1X.copy(), P1Y.copy()
    if add_point:
        X, y = np.r_[X, [[px, py]]], np.r_[y, 1]
    order = list(order) + ([4] if add_point else [])
    w_p, b_p = perceptron(X, y, order)
    svm = SVC(kernel="linear", C=1e6).fit(X, y)     # a very large C = hard margin
    w_s, b_s = svm.coef_[0], svm.intercept_[0]
    fig, axs = plt.subplots(1, 2, figsize=(11, 4.6))
    xl, yl = (-3, 5), (-4, 5)
    for ax, (w, b, title) in zip(axs, ((w_p, b_p, "perceptron"), (w_s, b_s, "SVM"))):
        d = y * (X @ w + b) / np.linalg.norm(w)
        if title == "SVM":
            draw_regions(ax, svm, xl, yl)
            ring(ax, X[svm.support_], label="support vectors")
        else:
            line(ax, w, b, xl, color=TEAL, lw=2.2)
        show_classes(ax, X, y, s=90)
        ax.set(xlim=xl, ylim=yl, xlabel="x₁", ylabel="x₂",
               title=f"{title}: w = ({w[0]:.2f}, {w[1]:.2f}), b = {b:.2f}\nsmallest distance {d.min():.2f}")
        ax.set_aspect("equal")
    axs[1].legend(loc="lower right", fontsize=8)
    plt.show()
    errors = int((svm.predict(X) != y).sum())
    print(f"SVM margin width 2/‖w‖ = {2 / np.linalg.norm(w_s):.3f}; "
          f"support vectors: {[f'x{i + 1}' for i in svm.support_]}")
    if errors:
        print(f"No error-free line exists: SVM leaves {errors} point(s) on the wrong side and the perceptron never stops (the plot shows its last line after 50 epochs).")
    if not add_point:
        assert np.allclose(w_s, [0.8, 0.4], atol=1e-3) and abs(b_s + 1) < 1e-3   # the lecture result


controls(section1,
         order=menu("perceptron order", [("x₁, x₂, x₃, x₄", (0, 1, 2, 3)), ("x₂, x₁, x₃, x₄", (1, 0, 2, 3)),
                                                                ("x₃, x₁, x₂, x₄", (2, 0, 1, 3))], (0, 1, 2, 3)),
         add_point=check("add a fifth point (+1)"),
         px=fslider("fifth point x₁", 0.5, -2, 4, 0.25), py=fslider("fifth point x₂", 1.5, -3, 4, 0.25))

**✏️ Task 1**

* Change the order for the perceptron. Which line has the larger smallest distance – any of the perceptron lines or the SVM line?
* Add the fifth point at (3, 4), far behind the support vectors. Does the SVM line change? And at (0.5, 1.5)?
* Move the fifth point to (−0.5, 1), between the two points of class −1. What happens to the margin? Then move it to (−0.5, 0): what do the message under the plot and the perceptron panel show, and why?

## 2. Soft margin and parameter C

**Goal of the method:** allow points inside the band or on the wrong side, but penalise every violation: minimise
½‖w‖² + C·Σξᵢ, where the slack ξᵢ = max(0, 1 − yᵢsᵢ). A small C means a wide band and many support vectors; a large C a narrow band shaped by single points.

**Controls:** C on a logarithmic scale, the overlap of the classes and the seed of the data. Under the plot: margin width, number of support vectors,
training errors and the cross-validated accuracy (5 folds).

In [ ]:
def section2(C, spread, seed):
    X, y = blobs(seed, 40, spread)
    m = SVC(kernel="linear", C=C).fit(X, y)
    xi = np.maximum(0, 1 - y * m.decision_function(X))   # slack variables
    fig, ax = plt.subplots(figsize=(6.2, 5.6))
    draw_regions(ax, m, (-4.5, 4.5), (-4.5, 4.5))
    show_classes(ax, X, y)
    ring(ax, X[(xi > 1e-6) & (xi < 1)], color=TEAL, label="0 < ξ < 1")
    ring(ax, X[xi >= 1], color=ORANGE, label="ξ ≥ 1")
    ax.set(xlim=(-4.5, 4.5), ylim=(-4.5, 4.5), xlabel="x₁", ylabel="x₂", title=f"C = {C:.3g}")
    ax.set_aspect("equal")
    ax.legend(loc="upper left", fontsize=8)
    plt.show()
    cv = cross_val_score(SVC(kernel="linear", C=C), X, y, cv=5).mean()
    print(f"margin width {2 / np.linalg.norm(m.coef_[0]):.2f} | support vectors {len(m.support_)} of {len(y)} | "
          f"training errors {int((m.predict(X) != y).sum())} | Σξ = {xi.sum():.1f} | CV accuracy {cv:.3f}")


controls(section2, C=logslider("C", 1.0, -3, 3), spread=fslider("class spread", 1.1, 0.5, 2.0, 0.1),
         seed=widgets.IntSlider(value=13, min=0, max=30, description="seed", continuous_update=False, style=STYLE, layout=WIDE))

**✏️ Task 2**

* Move C from 0.001 to 1000. How do the margin width and the number of support vectors change? Does the number of training errors change as much?
* For which C is the cross-validated accuracy best? Is it the C with the fewest training errors?
* Reduce the spread to 0.5 (separable classes). From which C on is the result the same as with the hard margin?

## 3. Hinge loss and other losses

**Goal:** compare how the losses from lectures 09 and 10 penalise a sample depending on y·s (score signed by the class).
Hinge loss max(0, 1 − y·s) is zero only from y·s ≥ 1; the perceptron loss max(0, −y·s) already from y·s > 0;
the logarithmic loss log₂(1 + e^(−y·s)) is never exactly zero.

**Controls:** the value y·s of one sample; the table shows the three losses for it.

In [ ]:
def section3(ys):
    t = np.linspace(-3, 3, 400)
    losses = {"hinge": lambda v: np.maximum(0, 1 - v), "perceptron": lambda v: np.maximum(0, -v),
              "logarithmic": lambda v: np.log2(1 + np.exp(-v))}
    fig, ax = plt.subplots(figsize=(7, 4.2))
    ax.plot(t, (t <= 0).astype(float), color=GREY, label="error count (0/1)")
    for (name, f), col, ls in zip(losses.items(), (BLUE, TEAL, ORANGE), ("-", "--", "-.")):
        ax.plot(t, f(t), color=col, ls=ls, lw=2.5, label=name)
        ax.scatter([ys], [f(ys)], color=col, s=60, zorder=4)
    ax.axvline(ys, color="black", lw=1, ls=":")
    ax.set(xlabel="y · s", ylabel="loss", ylim=(-0.1, 4))
    ax.legend(fontsize=9)
    plt.show()
    display(pd.DataFrame({name: [round(float(f(ys)), 3)] for name, f in losses.items()}, index=[f"y·s = {ys:g}"]))


controls(section3, ys=fslider("y · s", 0.5, -3, 3, 0.25))

**✏️ Task 3**

* Find the values of y·s for which the hinge loss is 0, 0.5 and 2 (the lecture example P3). What does the perceptron loss give for them?
* A sample with y·s = 0.5 is classified correctly. Which losses still penalise it and why does SVM want that?
* For y·s = −3: which loss grows fastest? Compare with squared error, which would be 16 for the distance 4 from the target.

## 4. Kernels: polynomial and RBF

**Goal of the method:** replace the dot product by a kernel K(x, z) and get a nonlinear boundary without computing new features.
Polynomial kernel (γ·x·z + 1)^d gives a boundary that is a polynomial of degree d; the RBF kernel exp(−γ‖x − z‖²) gives a boundary
of any shape, and γ sets how far the influence of one support vector reaches.

**Controls:** data (half-moons, circles, XOR), kernel, C, γ and the degree d. Under the plot: cross-validated accuracy and the number of support vectors.

In [ ]:
def dataset(name, seed=4):
    if name == "moons":
        X, y = make_moons(n_samples=200, noise=0.2, random_state=seed)
    elif name == "circles":
        X, y = make_circles(n_samples=200, factor=0.45, noise=0.09, random_state=seed)
        X = X * 1.6
    else:                                              # XOR: four clouds in the corners
        g = np.random.default_rng(seed)
        X = g.normal(0, 0.45, (200, 2)) + g.choice([-1, 1], (200, 2))
        y = (np.sign(X[:, 0]) != np.sign(X[:, 1])).astype(int)
    return X, np.where(y == 1, 1, -1)


def section4(data, kernel, C, gamma, degree):
    X, y = dataset(data)
    m = SVC(kernel=kernel, C=C, gamma=gamma, degree=degree, coef0=1)
    acc = cross_val_score(m, X, y, cv=5).mean()
    m.fit(X, y)
    fig, ax = plt.subplots(figsize=(6.2, 5.2))
    lo, hi = X.min(0) - 0.5, X.max(0) + 0.5
    draw_regions(ax, m, (lo[0], hi[0]), (lo[1], hi[1]), margins=False)
    show_classes(ax, X, y, s=25)
    ring(ax, X[m.support_], s=70)
    ax.set(xlim=(lo[0], hi[0]), ylim=(lo[1], hi[1]), xlabel="x₁", ylabel="x₂",
           title=f"{kernel}, C = {C:.3g}, γ = {gamma:.3g}" + (f", d = {degree}" if kernel == "poly" else ""))
    plt.show()
    print(f"CV accuracy {acc:.3f} | support vectors {len(m.support_)} of {len(y)}")


controls(section4, data=menu("data", [("half-moons", "moons"), ("circles", "circles"), ("XOR", "xor")], "moons"),
         kernel=menu("kernel", [("linear", "linear"), ("polynomial", "poly"), ("RBF", "rbf")], "rbf"),
         C=logslider("C", 1.0, -2, 3), gamma=logslider("γ", 1.0, -2, 2),
         degree=widgets.IntSlider(value=3, min=1, max=6, description="degree d", continuous_update=False, style=STYLE, layout=WIDE))

**✏️ Task 4**

* For every data set compare the linear, polynomial (d = 2 and 3) and RBF kernels. Which degree is enough for the circles, which for XOR, and why?
* RBF on the half-moons: move γ from 0.01 to 100 at C = 1. Where is the accuracy best, and what does the boundary look like at both ends?
* Choose γ = 30. Can you make the result better only by changing C? In which direction?

## 5. Scaling of features

**Goal:** see why SVM needs features on comparable scales. The margin and the RBF kernel measure distances; a feature with a large range dominates them.

**Controls:** the factor by which the second feature is multiplied (a change of units) and standardisation on or off. C = 1 and γ = 1 stay fixed.

In [ ]:
def section5(factor, scale):
    X, y = dataset("moons", seed=7)
    X = X * np.array([1.0, factor])                    # the same data in other units
    m = make_pipeline(StandardScaler(), SVC(C=1, gamma=1)) if scale else SVC(C=1, gamma=1)
    acc = cross_val_score(m, X, y, cv=5).mean()
    m.fit(X, y)
    fig, ax = plt.subplots(figsize=(6.2, 5))
    pad = (X.max(0) - X.min(0)) * 0.08
    xl, yl = (X[:, 0].min() - pad[0], X[:, 0].max() + pad[0]), (X[:, 1].min() - pad[1], X[:, 1].max() + pad[1])
    draw_regions(ax, m, xl, yl, margins=False)
    show_classes(ax, X, y, s=20)
    ax.set(xlim=xl, ylim=yl, xlabel="x₁", ylabel=f"x₂ × {factor:g}",
           title=("standardised" if scale else "unscaled") + f" · CV accuracy {acc:.3f}")
    plt.show()


controls(section5, factor=widgets.SelectionSlider(options=[0.01, 0.1, 1, 10, 30, 100], value=30, description="factor of x₂",
                                                  continuous_update=False, style=STYLE, layout=WIDE),
         scale=check("standardise features"))

**✏️ Task 5**

* Without scaling, move the factor from 1 to 100. How does the boundary change and why? What happens with the factor 0.01?
* Tick standardisation. Does the factor still matter? Why?
* Values of C and γ were tuned on data in one unit. Can you use them after a change of units without standardisation?

## 6. Regression with the ε-tube (SVR)

**Goal of the method:** a regression function that ignores deviations smaller than ε and penalises larger ones linearly.
Points on the edge of the tube or outside are the support vectors. The shaded band on the right is outside the training data:
there you see how each kernel **extrapolates**.

**Controls:** kernel, ε, C, γ and the visible range.

In [ ]:
g6 = np.random.default_rng(3)
x6 = np.sort(g6.uniform(0, 5, 60))                    # training inputs from 0 to 5
y6 = 0.6 * x6 + np.sin(1.6 * x6) + g6.normal(0, 0.15, x6.size)   # trend plus a wave plus noise


def section6(kernel, eps, C, gamma, extend):
    m = SVR(kernel=kernel, epsilon=eps, C=C, gamma=gamma, degree=2, coef0=1).fit(x6.reshape(-1, 1), y6)
    t = np.linspace(0, 8 if extend else 5, 400)
    f = m.predict(t.reshape(-1, 1))
    fig, ax = plt.subplots(figsize=(9, 4.2))
    if extend:
        ax.axvspan(5, 8, color=GREY, alpha=0.15)
        ax.plot(t[t >= 5], 0.6 * t[t >= 5] + np.sin(1.6 * t[t >= 5]), color=GREY, ls=":", label="true function")
    ax.fill_between(t, f - eps, f + eps, color=PALE_B, label="ε-tube")
    ax.plot(t, f, color=BLUE, lw=2.5, label="model")
    ax.scatter(x6, y6, color=ORANGE, s=22, zorder=3)
    ax.scatter(x6[m.support_], y6[m.support_], s=90, facecolors="none", edgecolors="black", zorder=4, label="support vectors")
    ax.set(xlabel="x", ylabel="y", title=f"{kernel}, ε = {eps:g} · {len(m.support_)} support vectors of {x6.size}")
    ax.legend(loc="upper left", fontsize=8)
    plt.show()
    print(f"training MAE {np.mean(np.abs(m.predict(x6.reshape(-1, 1)) - y6)):.3f}")


controls(section6, kernel=menu("kernel", [("RBF", "rbf"), ("linear", "linear"), ("polynomial d = 2", "poly")], "rbf"),
         eps=fslider("ε", 0.2, 0.0, 1.0, 0.05), C=logslider("C", 10.0, -1, 3), gamma=logslider("γ", 1.0, -2, 2),
         extend=check("show 5 to 8 (extrapolation)"))

**✏️ Task 6**

* Move ε from 0 to 1. How does the number of support vectors change? At which ε does the model stop following the wave?
* Tick the extrapolation range. What does the RBF model predict at x = 8? And the linear one? Which is closer to the true function, and why?
* Can a smaller γ improve the RBF extrapolation? What does it cost inside the data?

## 7. One-class SVM

**Goal of the method:** learn a boundary around normal data only, without labelled anomalies. ν is roughly an upper bound on the share
of training samples left outside; γ sets how closely the boundary follows the data.

**Controls:** ν, γ and whether the training data contain a few faults. Crosses are new anomalies that the model never saw.

In [ ]:
g7 = np.random.default_rng(34)
normal7 = np.r_[g7.normal([0, 0], [1.0, 0.6], (150, 2)), g7.normal([2.6, 1.6], 0.5, (60, 2))]   # normal operation
new7 = g7.uniform([-4, -3.5], [5.5, 5], (40, 2))
new7 = new7[np.min(np.linalg.norm(new7[:, None] - normal7[None], axis=2), axis=1) > 0.9][:10]   # new anomalies
dirty7 = g7.uniform([-3, -3], [4, 4], (12, 2))         # faults hidden in the training data


def section7(nu, gamma, dirty):
    train = np.r_[normal7, dirty7] if dirty else normal7
    m = OneClassSVM(nu=nu, gamma=gamma).fit(train)
    xx, yy = np.meshgrid(np.linspace(-4, 5.5, 250), np.linspace(-3.5, 5, 250))
    zz = m.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    out = m.predict(train) == -1
    found = int((m.predict(new7) == -1).sum())
    fig, ax = plt.subplots(figsize=(6.4, 5.4))
    ax.contourf(xx, yy, zz, levels=[0, zz.max()], colors=[PALE_B])
    ax.contour(xx, yy, zz, levels=[0], colors="black", linewidths=2)
    ax.scatter(*train[~out].T, s=16, color=BLUE, label="training data")
    ax.scatter(*train[out].T, s=30, color=BLUE, edgecolor=ORANGE, lw=1.6, label="training sample outside")
    ax.scatter(*new7.T, marker="x", s=90, color=ORANGE, lw=2.5, label="new anomalies")
    ax.set(xlim=(-4, 5.5), ylim=(-3.5, 5), xlabel="x₁", ylabel="x₂",
           title=f"ν = {nu:g}, γ = {gamma:.3g} · {100 * out.mean():.0f} % outside · found {found}/{len(new7)}")
    ax.set_aspect("equal")
    ax.legend(loc="upper left", fontsize=8)
    plt.show()


controls(section7, nu=fslider("ν", 0.05, 0.01, 0.5, 0.01), gamma=logslider("γ", 0.2, -2, 1),
         dirty=check("12 faults in the training data"))

**✏️ Task 7**

* Move ν from 0.01 to 0.5. Compare the share of training samples outside with ν. How many new anomalies are found at both ends?
* Raise γ to 5. What does the boundary look like and what would happen with new normal data that lie between the training points?
* Tick the faults in the training data. How does the boundary change at ν = 0.02? Which ν helps, and what does this say about cleaning the training data?

## 8. Blessing and curse of dimensionality

**Goal:** see why a mapping into many features makes linear separation easy – and why that alone is not enough.
Left: n random points in d dimensions with **random** classes; the share of class assignments that some hyperplane separates without error.
The line is Cover's theorem P = 2^(1−n)·Σ_{k=0}^{d} C(n−1, k), the dots a simulation (exact test by a linear program).
Right: 30 random points in 2D with random classes and an SVM with a polynomial kernel of degree p and a very large C (practically a hard margin).

**Controls:** dimension d and the number of points n (left); polynomial degree p (right). Under the plot: training accuracy,
cross-validated accuracy and the number of support vectors.

In [ ]:
from math import comb
from scipy.optimize import linprog                  # exact test of linear separability


def cover(n, d):
    return 2 * sum(comb(n - 1, k) for k in range(d + 1)) / 2 ** n   # hyperplane with a bias in d dimensions


def separable(X, y):
    s = np.where(y == 1, 1.0, -1.0)
    A = -s[:, None] * np.c_[X, np.ones(len(X))]      # find w, b with y(w·x + b) ≥ 1
    return linprog(np.zeros(X.shape[1] + 1), A_ub=A, b_ub=-np.ones(len(X)), bounds=[(None, None)] * (X.shape[1] + 1), method="highs").status == 0


g8 = np.random.default_rng(8)
X8 = g8.uniform(-1, 1, (30, 2)); y8 = np.where(g8.random(30) < 0.5, 1, -1)   # random points, random classes


def section8(d, n, p):
    fig, axs = plt.subplots(1, 2, figsize=(12, 4.4))
    ax = axs[0]
    ns = np.arange(1, 4 * (d + 1) + 1)
    ax.plot(ns, [cover(int(k), d) for k in ns], color=BLUE, label="Cover's theorem")
    g = np.random.default_rng(d)
    sim = np.mean([separable(g.normal(size=(n, d)), np.where(g.random(n) < 0.5, 1, -1)) for _ in range(40)])
    ax.scatter([n], [sim], color=ORANGE, s=90, zorder=4, label=f"simulation n = {n}: {sim:.2f}")
    ax.axvline(2 * (d + 1), color=GREY, ls=":")
    ax.set(xlabel="number of points n", ylabel="separable share", ylim=(-0.05, 1.05), title=f"d = {d}")
    ax.legend(fontsize=9)
    ax = axs[1]
    m = SVC(kernel="poly", degree=p, coef0=1, gamma=1, C=1e6).fit(X8, y8)
    draw_regions(ax, m, (-1.1, 1.1), (-1.1, 1.1), margins=False)
    show_classes(ax, X8, y8, s=45)
    ax.set(xlim=(-1.1, 1.1), ylim=(-1.1, 1.1), title=f"random classes, polynomial degree {p}")
    ax.set_aspect("equal")
    plt.show()
    train = m.score(X8, y8)
    cv = cross_val_score(SVC(kernel="poly", degree=p, coef0=1, gamma=1, C=1e6), X8, y8, cv=5).mean()
    print(f"training accuracy {train:.2f} | CV accuracy {cv:.2f} | support vectors {len(m.support_)} of 30")


controls(section8, d=widgets.SelectionSlider(options=[1, 2, 5, 10, 20], value=2, description="d", continuous_update=False, style=STYLE, layout=WIDE),
         n=widgets.IntSlider(value=6, min=2, max=60, description="n", continuous_update=False, style=STYLE, layout=WIDE),
         p=widgets.IntSlider(value=1, min=1, max=8, description="degree p", continuous_update=False, style=STYLE, layout=WIDE))

**✏️ Task 8**

* For d = 2, find n where the simulated share is about one half. Check it with the rule n = 2(d + 1). Repeat for d = 10.
* On the right, raise the degree p from 1 to 8. From which degree are all 30 random points classified correctly? What is the cross-validated accuracy then, and why?
* The classes are random, yet from some degree on the training error is zero. What does zero training error mean in a space with many features, and what must be checked besides it (lecture: curse of dimensionality and the margin)?